In [1]:
import pandas as pd
import numpy as np
# XGBClassifier
from xgboost import XGBClassifier

In [ ]:
orb = pd.read_json("https://minorplanetcenter.net/Extended_Files/mpcorb_extended.json.gz",compression='gzip')
orb = orb[orb["Num_opps"] == 1]


In [ ]:
# Mislinkage filtering by bringing in a training dataset and training a probabilistic binary classifier
mislinkage_training = pd.read_csv("mislinkage training.csv")
print(len(mislinkage_training))
desigs_in_training = mislinkage_training["Principal_desig"]
mislinkage_training = mislinkage_training[["Num_obs", "rms", "Arc_length", "likely_or_possible_mislinkage"]]
X_df_misl = mislinkage_training.drop(columns=["likely_or_possible_mislinkage"])
y_df_misl = mislinkage_training["likely_or_possible_mislinkage"]
xgb_misl = XGBClassifier()
xgb_misl.fit(X_df_misl, y_df_misl)
orb["mislinkage_pred"] = xgb_misl.predict_proba(orb[X_df_misl.columns])[:,1]

2224


In [ ]:
orb[(orb["Principal_desig"].str[0:4] == "2023") & (orb["mislinkage_pred"] > 0.7)][["Num_obs", "rms", "Arc_length", "Principal_desig"]].to_clipboard(index=False)

In [ ]:
list_desigs = ["2017 VJ70",'2025 XL21']
orb[(orb["Principal_desig"].isin(list_desigs)) ][["Num_obs", "rms", "Arc_length", "Principal_desig"]].to_clipboard(index=False)

In [ ]:
orb = orb[~orb["Principal_desig"].isin(desigs_in_training)]
orb.reset_index(drop=True, inplace=True)

In [ ]:
from modAL.models import ActiveLearner
from modAL.uncertainty import uncertainty_sampling
import pyperclip

# Set up the ActiveLearner
learner = ActiveLearner(
    estimator=XGBClassifier(),
    query_strategy=uncertainty_sampling,
    X_training=X_df_misl.to_numpy(),
    y_training=y_df_misl.to_numpy()
)

new_training = pd.DataFrame()

same = ''
while same != 'q':
	query_idx, query_inst = learner.query(orb[X_df_misl.columns].to_numpy())
	example = orb.iloc[query_idx]
	print(example["Principal_desig"])
	pyperclip.copy(example["Principal_desig"].to_string(index=False))
	same = input("Mislinkage?")
	if same != '' and same != 'q':
		y_new = int(same)
		learner.teach(example[X_df_misl.columns].to_numpy(), [y_new])
		new_row = orb.iloc[query_idx].copy()
		new_row['mislinkage'] = y_new
		new_training = pd.concat([new_training, new_row], ignore_index=True)
		print("Added to training set.")

		orb = orb.drop(index=query_idx)
		orb.reset_index(drop=True, inplace=True)
	if same == '':
		orb = orb.drop(index=query_idx)
		orb.reset_index(drop=True, inplace=True)

110264    2022 RN74
Name: Principal_desig, dtype: object


In [ ]:
new_training[["Num_obs", "rms", "Arc_length", "mislinkage", "Principal_desig"]].to_clipboard(index=False)

KeyError: "None of [Index(['Num_obs', 'rms', 'Arc_length', 'mislinkage', 'Principal_desig'], dtype='object')] are in the [columns]"